# مرصد الأخبار: أدوات مجانية على Google Colab

دفتر اختياري يعمل على Colab المجاني، ولا يحتاج API مدفوع أو اشتراك Colab Pro. يزيل تكرار الأخبار ويجمعها مبدئياً حسب تشابه النص. النتائج اقتراحات آلية وليست إثباتاً لعلاقة بين الأحداث.

In [ ]:
!pip -q install pandas scikit-learn
import pandas as pd
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
print('Rows:', len(df), 'Columns:', list(df.columns))


In [ ]:
# إزالة التكرار بالرابط أولاً، ثم بالعنوان عند غياب الرابط
if 'url' in df.columns:
    df = df.drop_duplicates(subset=['url'], keep='first')
if 'title' in df.columns:
    df = df.drop_duplicates(subset=['title'], keep='first')
df = df.fillna('')
print('After deduplication:', len(df))
display(df.head(10))


In [ ]:
# تجميع استكشافي خفيف من العنوان والملخص
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
import numpy as np
text_cols = [c for c in ['title', 'summary'] if c in df.columns]
if len(df) >= 3 and text_cols:
    corpus = df[text_cols].astype(str).agg(' '.join, axis=1).tolist()
    X = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=1).fit_transform(corpus)
    k = min(8, max(2, int(np.sqrt(len(df) / 2))))
    df['auto_cluster'] = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3).fit_predict(X)
    print('Clusters:', k)
else:
    df['auto_cluster'] = 0
display(df.head(20))


In [ ]:
out = 'marsad_colab_cleaned.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')
files.download(out)
